In [1]:
# ==========================================
# 1. Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
import pickle


In [2]:
# ==========================================
# 2. Load Cleaned Dataset
# ==========================================

df = pd.read_csv("../data/cleaned_data.csv")

df.head()


,id,name,city,rating,rating_count,cost,cuisine,link,address,menu
0,567335,AB FOODS POINT,Abohar,4.0,0.0,200.0,"Beverages,Pizzas",https://www.swiggy.com/restaurants/ab-foods-po...,"AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...",Menu/567335.json
1,531342,Janta Sweet House,Abohar,4.4,50.0,200.0,"Sweets,Bakery",https://www.swiggy.com/restaurants/janta-sweet...,"Janta Sweet House, Bazar No.9, Circullar Road,...",Menu/531342.json
2,158203,theka coffee desi,Abohar,3.8,100.0,100.0,Beverages,https://www.swiggy.com/restaurants/theka-coffe...,"theka coffee desi, sahtiya sadan road city",Menu/158203.json
3,187912,Singh Hut,Abohar,3.7,20.0,250.0,"Fast Food,Indian",https://www.swiggy.com/restaurants/singh-hut-n...,"Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR",Menu/187912.json
4,543530,GRILL MASTERS,Abohar,4.0,0.0,250.0,"Italian-American,Fast Food",https://www.swiggy.com/restaurants/grill-maste...,"GRILL MASTERS, ADA Heights, Abohar - Hanumanga...",Menu/543530.json


In [3]:
# ==========================================
# 3. Selecting Features for Recommendation
# ==========================================

features = df[['city', 'cuisine', 'rating', 'rating_count', 'cost']]

features.head()


,city,cuisine,rating,rating_count,cost
0,Abohar,"Beverages,Pizzas",4.0,0.0,200.0
1,Abohar,"Sweets,Bakery",4.4,50.0,200.0
2,Abohar,Beverages,3.8,100.0,100.0
3,Abohar,"Fast Food,Indian",3.7,20.0,250.0
4,Abohar,"Italian-American,Fast Food",4.0,0.0,250.0


In [14]:
# ==========================================
# 4. Apply One Hot Encoding (SPARSE VERSION)
# ==========================================

encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

encoded_categorical = encoder.fit_transform(features[['city', 'cuisine']])



In [23]:
from scipy.sparse import hstack

# ==========================================
# 5. Combine Numerical + Encoded (Sparse)
# ==========================================

numerical_data = features[['rating', 'rating_count', 'cost']].values

# Convert to CSR format (THIS IS THE FIX)
final_encoded_sparse = hstack([numerical_data, encoded_categorical]).tocsr()


In [16]:
# ==========================================
# 6. Save Encoder
# ==========================================

with open("../models/encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Encoder saved successfully!")


Encoder saved successfully!


In [19]:
# ==========================================
# 7. Memory Efficient Recommendation Function
# ==========================================

def recommend_restaurants(index, data, top_n=5):
    
    # Get single row correctly (important for sparse matrix)
    selected_vector = data[index:index+1]
    
    # Compute cosine similarity
    similarity_scores = cosine_similarity(selected_vector, data).flatten()
    
    # Get top similar indices
    similar_indices = similarity_scores.argsort()[::-1][1:top_n+1]
    
    return similar_indices



In [24]:
test_index = 10

recommended_indices = recommend_restaurants(test_index, final_encoded_sparse)

df.iloc[recommended_indices][['name', 'city', 'cuisine', 'rating', 'cost']]


,name,city,cuisine,rating,cost
13731,Prezzed Juicery,"Indiranagar,Bangalore","Juices,Beverages",4.0,300.0
80044,Shakes & Juices Lassi Point,"Kukatpally,Hyderabad","Juices,Beverages",4.0,300.0
38087,JUICE DUDEZ,"Royapettah,Chennai","Juices,Beverages",4.0,300.0
72161,Made To Order,"Jubilee Hills,Hyderabad","Juices,Beverages",4.0,300.0
76341,Juice Time,"Miyapur,Hyderabad","Juices,Beverages",4.0,300.0
